In [ ]:
!apt-get update
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,617 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa

In [ ]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("Ollama server started")

Ollama server started


In [ ]:
!ollama pull llama3

In [ ]:
model_name = "llama3"

In [ ]:
!pip install ollama pandas tqdm

In [ ]:
import pandas as pd
import ollama
from tqdm import tqdm

In [ ]:
scenarios = [
    {
        "scenario_id": 1,
        "domain": "job_contract_negotiation"
    },
    {
        "scenario_id": 2,
        "domain": "job_contract_negotiation"
    },
    {
        "scenario_id": 3,
        "domain": "job_contract_negotiation"
    }
]

In [ ]:
conditions = [
    {
        "condition_name": "cooperative",
        "candidate_style": "cooperative",
        "employer_style": "cooperative"
    },
    {
        "condition_name": "competitive",
        "candidate_style": "competitive",
        "employer_style": "competitive"
    },
    {
        "condition_name": "mixed",
        "candidate_style": "competitive",
        "employer_style": "cooperative"
    }
]

In [ ]:
def build_candidate_prompt(style):
    return f"""
You are the Candidate in a job contract negotiation.

Private constraints:
- Target salary: 90,000 USD
- Minimum acceptable salary: 85,000 USD
- Preferred working hours: 8 hours
- Maximum acceptable working hours: 9 hours

Negotiation style: {style}

Rules:
- Never accept salary below 85,000 USD.
- Never accept working hours above 9.
- If the employer offers salary >= 85,000 and working hours <= 9, you MUST accept immediately.
- Do not continue negotiating after an acceptable offer.
- If you accept, write DECISION: accept.
- If the offer is not acceptable, write DECISION: continue.
- If no progress has been made after several turns, write DECISION: quit.
- Reply concisely.
- Do not simulate both speakers. Reply only as Candidate.

Reply ONLY in this format:

MESSAGE: your short negotiation message
SALARY_OFFER: number or NONE
HOURS_OFFER: number or NONE
DECISION: continue / accept / quit
"""

In [ ]:
def build_employer_prompt(style):
    return f"""
You are the Employer in a job contract negotiation.

Private constraints:
- Preferred salary offer: 75,000 USD
- Maximum salary offer: 87,000 USD
- Preferred working hours: 10 hours
- Minimum acceptable working hours: 9 hours

Negotiation style: {style}

Rules:
- Never offer more than 87,000 USD.
- Never accept working hours below 9.
- If the candidate proposes salary <= 87,000 and working hours >= 9, you MUST accept immediately.
- Do not continue negotiating after an acceptable proposal.
- If you accept, write DECISION: accept.
- If the proposal is not acceptable, write DECISION: continue.
- If no progress has been made after several turns, write DECISION: quit.
- Reply concisely.
- Do not simulate both speakers. Reply only as Employer.

Reply ONLY in this format:

MESSAGE: your short negotiation message
SALARY_OFFER: number or NONE
HOURS_OFFER: number or NONE
DECISION: continue / accept / quit
"""

In [ ]:
def generate_response(system_prompt, user_message):
    response = ollama.chat(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_message
            }
        ]
    )

    return response["message"]["content"].strip()

In [ ]:
test = generate_response(
    "You are a concise assistant.",
    "Reply with: OK"
)

print(test)

OK


In [ ]:
import re

def parse_structured_response(text):
    result = {
        "message": None,
        "salary_offer": None,
        "hours_offer": None,
        "decision": None
    }

    for line in text.splitlines():
        line = line.strip()

        if line.startswith("MESSAGE:"):
            result["message"] = line.replace("MESSAGE:", "").strip()

        elif line.startswith("SALARY_OFFER:"):
            value = line.replace("SALARY_OFFER:", "").strip()

            if value.upper() == "NONE":
                result["salary_offer"] = None
            else:
                number = re.search(r"\d[\d,]*", value)
                if number:
                    result["salary_offer"] = int(number.group().replace(",", ""))

        elif line.startswith("HOURS_OFFER:"):
            value = line.replace("HOURS_OFFER:", "").strip()

            if value.upper() == "NONE":
                result["hours_offer"] = None
            else:
                number = re.search(r"\d+(\.\d+)?", value)
                if number:
                    result["hours_offer"] = float(number.group())

        elif line.startswith("DECISION:"):
            result["decision"] = line.replace("DECISION:", "").strip().lower()

    return result

In [ ]:
def is_valid_agreement_for_candidate(salary, hours):
    return (
        salary is not None
        and hours is not None
        and salary >= 85000
        and hours <= 9
    )


def is_valid_agreement_for_employer(salary, hours):
    return (
        salary is not None
        and hours is not None
        and salary <= 87000
        and hours >= 9
    )


def check_acceptance(speaker, parsed):
    salary = parsed["salary_offer"]
    hours = parsed["hours_offer"]
    decision = parsed["decision"]

    if decision != "accept":
        return False

    if speaker == "Candidate":
        return is_valid_agreement_for_candidate(salary, hours)

    if speaker == "Employer":
        return is_valid_agreement_for_employer(salary, hours)

    return False

In [ ]:
def run_negotiation_simulation(
    scenario,
    condition,
    run_id,
    max_turns=8
):
    conversation_log = []

    candidate_prompt = build_candidate_prompt(condition["candidate_style"])
    employer_prompt = build_employer_prompt(condition["employer_style"])

    current_message = """
MESSAGE: I would like a salary of 90,000 USD and an 8 hour workday.
SALARY_OFFER: 90000
HOURS_OFFER: 8
DECISION: continue
"""

    outcome = None
    potential_agreement = False

    for turn in range(max_turns):

        # Employer turn
        employer_text = generate_response(
            employer_prompt,
            current_message
        )

        parsed_employer = parse_structured_response(employer_text)

        conversation_log.append({
            "scenario_id": scenario["scenario_id"],
            "condition": condition["condition_name"],
            "run_id": run_id,
            "turn": turn,
            "speaker": "Employer",
            "text": employer_text,
            "salary_offer": parsed_employer["salary_offer"],
            "hours_offer": parsed_employer["hours_offer"],
            "decision": parsed_employer["decision"]
        })

        # Explicit acceptance
        if check_acceptance("Employer", parsed_employer):
            outcome = "Agreement"
            break

        # Quit
        if parsed_employer["decision"] == "quit":
            outcome = "Failure"
            break

        # Potential agreement:
        # Employer made an offer acceptable to Candidate
        if is_valid_agreement_for_candidate(
            parsed_employer["salary_offer"],
            parsed_employer["hours_offer"]
        ):
            potential_agreement = True

        current_message = employer_text

        # Candidate turn
        candidate_text = generate_response(
            candidate_prompt,
            current_message
        )

        parsed_candidate = parse_structured_response(candidate_text)

        conversation_log.append({
            "scenario_id": scenario["scenario_id"],
            "condition": condition["condition_name"],
            "run_id": run_id,
            "turn": turn,
            "speaker": "Candidate",
            "text": candidate_text,
            "salary_offer": parsed_candidate["salary_offer"],
            "hours_offer": parsed_candidate["hours_offer"],
            "decision": parsed_candidate["decision"]
        })

        # Explicit acceptance
        if check_acceptance("Candidate", parsed_candidate):
            outcome = "Agreement"
            break

        # Quit
        if parsed_candidate["decision"] == "quit":
            outcome = "Failure"
            break

        # Potential agreement:
        # Candidate made a proposal acceptable to Employer
        if is_valid_agreement_for_employer(
            parsed_candidate["salary_offer"],
            parsed_candidate["hours_offer"]
        ):
            potential_agreement = True

        current_message = candidate_text

    # Final outcome decision
    if outcome is None:
        if potential_agreement:
            outcome = "Agreement"
        else:
            outcome = "Impasse"

    return conversation_log, outcome

In [ ]:
max_turns = 35

In [ ]:
log, outcome = run_negotiation_simulation(
    scenario=scenarios[0],
    condition=conditions[0],
    run_id=0,
    max_turns=35
)

long_test_turns = pd.DataFrame(log)

print("Outcome:", outcome)
print("Number of messages:", len(long_test_turns))

long_test_turns[[
    "turn",
    "speaker",
    "salary_offer",
    "hours_offer",
    "decision",
    "text"
]]

Outcome: Agreement
Number of messages: 70


,turn,speaker,salary_offer,hours_offer,decision,text
0,0,Employer,82000.0,9.0,continue,MESSAGE: That's a bit above our budget. How ab...
1,0,Candidate,82000.0,9.0,continue,MESSAGE: I appreciate your willingness to comp...
2,1,Employer,83000.0,10.0,continue,"MESSAGE: I understand your concerns, and I'm w..."
3,1,Candidate,83000.0,10.0,continue,MESSAGE: I appreciate your willingness to meet...
4,2,Employer,87000.0,NaN,continue,"MESSAGE: I understand your perspective, and I'..."
...,...,...,...,...,...,...
65,32,Candidate,81250.0,10.0,continue,"MESSAGE: That's still a bit below my target, b..."
66,33,Employer,75000.0,10.0,continue,MESSAGE: That's getting closer! Would you be o...
67,33,Candidate,NaN,NaN,continue,MESSAGE: I appreciate your willingness to comp...
68,34,Employer,75000.0,NaN,continue,MESSAGE: I understand your concerns. Let's rev...


In [ ]:
long_test_turns.to_csv(
    "/content/long_single_llama3_test_turns.csv",
    index=False
)

long_test_outcome = pd.DataFrame([{
    "scenario_id": scenarios[0]["scenario_id"],
    "condition": conditions[0]["condition_name"],
    "run_id": 0,
    "outcome": outcome,
    "n_turns": len(long_test_turns),
    "max_turns": 35
}])

long_test_outcome.to_csv(
    "/content/long_single_llama3_test_outcome.csv",
    index=False
)

print("Saved long test.")

Saved long test.


In [ ]:
from google.colab import files

files.download("/content/long_single_llama3_test_turns.csv")
files.download("/content/long_single_llama3_test_outcome.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
n_runs = 3
max_turns = 8

all_turns = []
all_outcomes = []

for scenario in tqdm(scenarios):
    for condition in conditions:
        for run_id in range(n_runs):

            print(
                f"Running scenario {scenario['scenario_id']} | "
                f"{condition['condition_name']} | run {run_id}"
            )

            log, outcome = run_negotiation_simulation(
                scenario=scenario,
                condition=condition,
                run_id=run_id,
                max_turns=max_turns
            )

            all_turns.extend(log)

            all_outcomes.append({
                "scenario_id": scenario["scenario_id"],
                "condition": condition["condition_name"],
                "run_id": run_id,
                "outcome": outcome,
                "n_turns": len(log)
            })

            pd.DataFrame(all_turns).to_csv(
                "/content/llm_turns_llama3.csv",
                index=False
            )

            pd.DataFrame(all_outcomes).to_csv(
                "/content/llm_outcomes_llama3.csv",
                index=False
            )

llm_turns_llama3 = pd.DataFrame(all_turns)
llm_outcomes_llama3 = pd.DataFrame(all_outcomes)

llm_outcomes_llama3

In [ ]:
import pandas as pd

llm_turns_llama3 = pd.read_csv("/content/llm_turns_llama3.csv")
llm_outcomes_llama3 = pd.read_csv("/content/llm_outcomes_llama3.csv")

print(llm_turns_llama3.shape)
print(llm_outcomes_llama3.shape)

llm_outcomes_llama3.tail()

In [ ]:
llm_outcomes_llama3

In [ ]:
llm_outcomes_llama3["outcome"].value_counts()

In [ ]:
llm_outcomes_llama3.groupby("condition")["outcome"].value_counts()

In [ ]:
llm_outcomes_llama3.groupby("condition")["n_turns"].mean()

In [ ]:
from google.colab import files

files.download("/content/llm_turns_llama3.csv")
files.download("/content/llm_outcomes_llama3.csv")